# Shared Train/Validation/Test Split for Predictive Models

In [1]:
from datetime import datetime

import polars as pl

from run_config import (
    PATHS,
    RUN_MODE,
    MODEL_START_DATE,
    MODEL_END_DATE,
)

## Train Test Split

The input and output paths are selected by `RUN_MODE` in `run_config.py`. Thus, both `sample` and `full` mode write to separate locations. The generated 70/15/15 train, validation and test files are shared inputs for all predictive models. The random split is grouped by calendar date, so all spatial units and time buckets of a day remain in the same split. The checks below verify disjoint dates and report complete-day counts plus zero- and positive-demand coverage for every partition.

In [2]:
DATASETS = (
    *PATHS.gold_1h_demand_hexagons.values(),
    PATHS.gold_1h_demand_census_tracts,
    PATHS.gold_1h_demand_community_areas,
    PATHS.gold_1h_demand_community_area_unfiltered,
    *PATHS.gold_24h_demand_hexagons.values(),
    PATHS.gold_24h_demand_census_tracts,
    PATHS.gold_24h_demand_community_areas,
    PATHS.gold_24h_demand_community_area_unfiltered,
    *PATHS.gold_4h_demand_hexagons.values(),
    PATHS.gold_4h_demand_census_tracts,
    PATHS.gold_4h_demand_community_areas,
    PATHS.gold_4h_demand_community_area_unfiltered,
)
OUTPUT_DIR = PATHS.train_test_dir
TARGET_COL = "trip_count"

SEED = 42
RANDOM = True

MODEL_START_TS = datetime.fromisoformat(MODEL_START_DATE)
MODEL_END_TS = datetime.fromisoformat(MODEL_END_DATE)
if MODEL_START_TS >= MODEL_END_TS:
    raise ValueError(
        f"MODEL_START_DATE must be before MODEL_END_DATE: "
        f"{MODEL_START_DATE} >= {MODEL_END_DATE}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run mode: {RUN_MODE}")
if RUN_MODE == "full":
    print(f"Model period: [{MODEL_START_DATE}, {MODEL_END_DATE})")
else:
    print("Model-period filter disabled in sample mode")
print(f"Inputs: {[path.name for path in DATASETS]}")
print(f"Output directory: {OUTPUT_DIR}")

Run mode: full
Model period: [2025-01-01T00:00:00, 2026-05-01T00:00:00)
Inputs: ['GOLD_1H_DEMAND_HEXAGON_7.parquet', 'GOLD_1H_DEMAND_HEXAGON_8.parquet', 'GOLD_1H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_1H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet', 'GOLD_24H_DEMAND_HEXAGON_7.parquet', 'GOLD_24H_DEMAND_HEXAGON_8.parquet', 'GOLD_24H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_24H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_24H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet', 'GOLD_4H_DEMAND_HEXAGON_7.parquet', 'GOLD_4H_DEMAND_HEXAGON_8.parquet', 'GOLD_4H_DEMAND_CENSUS_TRACTS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS.parquet', 'GOLD_4H_DEMAND_COMMUNITY_AREAS_UNFILTERED.parquet']
Output directory: /Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data


In [3]:
def create_splits(dataset_path):
    df_split = pl.scan_parquet(dataset_path)

    if RUN_MODE == "full":
        df_split = df_split.filter(
            (pl.col("datetime_hour") >= MODEL_START_TS)
            & (pl.col("datetime_hour") < MODEL_END_TS)
        )

    if RANDOM:
        # Keep every spatial unit and time bucket from the same calendar day
        # in one split. Unique dates are ordered reproducibly by their hash.
        # Validation and test receive exactly the same number of complete days.
        date_assignment = (
            df_split
            .select(pl.col("datetime_hour").dt.date().alias("_split_date"))
            .unique()
            .with_columns(
                pl.col("_split_date").hash(seed=SEED).alias("_split_order")
            )
            .sort(["_split_order", "_split_date"])
            .collect()
            .with_row_index("_date_rank")
        )
        n_dates = date_assignment.height
        n_holdout_dates = round(n_dates * 0.15)
        n_train_dates = n_dates - 2 * n_holdout_dates
        if n_train_dates <= 0 or n_holdout_dates <= 0:
            raise ValueError(f"Not enough dates for grouped 70/15/15 split: {n_dates}")

        date_assignment = (
            date_assignment
            .with_columns(
                pl.when(pl.col("_date_rank") < n_train_dates)
                .then(pl.lit("train"))
                .when(pl.col("_date_rank") < n_train_dates + n_holdout_dates)
                .then(pl.lit("val"))
                .otherwise(pl.lit("test"))
                .alias("_split")
            )
            .select(["_split_date", "_split"])
        )
        bucketed = (
            df_split
            .with_columns(
                pl.col("datetime_hour").dt.date().alias("_split_date")
            )
            .join(date_assignment.lazy(), on="_split_date", how="inner")
        )
        train = bucketed.filter(pl.col("_split") == "train")
        val = bucketed.filter(pl.col("_split") == "val")
        test = bucketed.filter(pl.col("_split") == "test")
        helper_columns = ["_split_date", "_split"]
        train = train.drop(helper_columns)
        val = val.drop(helper_columns)
        test = test.drop(helper_columns)
    else:
        train = df_split.filter(pl.col("datetime_hour") < pl.datetime(2025, 9, 1))
        val = df_split.filter(
            (pl.col("datetime_hour") >= pl.datetime(2025, 9, 1))
            & (pl.col("datetime_hour") < pl.datetime(2026, 1, 1))
        )
        test = df_split.filter(pl.col("datetime_hour") >= pl.datetime(2026, 1, 1))

    return df_split, train, val, test


split_results = {}
for dataset_path in DATASETS:
    df_split, train, val, test = create_splits(dataset_path)
    output_paths = {
        "train": OUTPUT_DIR / f"{dataset_path.stem}_TRAIN.parquet",
        "val": OUTPUT_DIR / f"{dataset_path.stem}_VAL.parquet",
        "test": OUTPUT_DIR / f"{dataset_path.stem}_TEST.parquet",
    }

    counts = {
        "total": df_split.select(pl.len()).collect().item(),
        "train": train.select(pl.len()).collect().item(),
        "val": val.select(pl.len()).collect().item(),
        "test": test.select(pl.len()).collect().item(),
    }
    if counts["total"] == 0:
        raise ValueError(f"Input dataset is empty: {dataset_path}")
    if counts["train"] + counts["val"] + counts["test"] != counts["total"]:
        raise ValueError(f"Split counts do not add up for {dataset_path}")

    split_dates = {
        name: frame.select(
            pl.col("datetime_hour").dt.date().alias("date")
        ).unique().collect()
        for name, frame in {"train": train, "val": val, "test": test}.items()
    }
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = split_dates[left].join(split_dates[right], on="date", how="inner")
        if overlap.height:
            raise ValueError(f"Date leakage between {left} and {right}: {overlap.height} dates")
    date_counts = {name: dates.height for name, dates in split_dates.items()}

    target_stats = {}
    for name, frame in {"train": train, "val": val, "test": test}.items():
        stats = frame.select(
            pl.col(TARGET_COL).is_null().sum().alias("null_targets"),
            (pl.col(TARGET_COL) == 0).sum().alias("zero_demand"),
            (pl.col(TARGET_COL) > 0).sum().alias("positive_demand"),
            pl.col(TARGET_COL).min().alias("min_demand"),
            pl.col(TARGET_COL).mean().alias("mean_demand"),
            pl.col(TARGET_COL).max().alias("max_demand"),
        ).collect().row(0, named=True)
        if stats["null_targets"]:
            raise ValueError(
                f"{dataset_path.name} {name} contains null target values"
            )
        if stats["min_demand"] < 0:
            raise ValueError(
                f"{dataset_path.name} {name} contains negative demand"
            )
        if stats["zero_demand"] == 0 or stats["positive_demand"] == 0:
            raise ValueError(
                f"{dataset_path.name} {name} must contain both zero- and "
                "positive-demand observations"
            )
        target_stats[name] = stats

    train.sink_parquet(output_paths["train"])
    val.sink_parquet(output_paths["val"])
    test.sink_parquet(output_paths["test"])
    split_results[dataset_path.stem] = {
        "counts": counts,
        "date_counts": date_counts,
        "target_stats": target_stats,
        "paths": output_paths,
    }

    shares = {name: round(count / counts["total"], 2) for name, count in counts.items() if name != "total"}
    print(f"{dataset_path.name}: {counts}, shares={shares}")
    print(f"Grouped split dates: {date_counts}")
    print(f"Target coverage: {target_stats}")
    print(f"Written: {output_paths}")

GOLD_1H_DEMAND_HEXAGON_7.parquet: {'total': 1897320, 'train': 1326168, 'val': 285576, 'test': 285576}, shares={'train': 0.7, 'val': 0.15, 'test': 0.15}
Grouped split dates: {'train': 339, 'val': 73, 'test': 73}
Target coverage: {'train': {'null_targets': 0, 'zero_demand': 1261315, 'positive_demand': 64853, 'min_demand': 0, 'mean_demand': 1.8778744472796811, 'max_demand': 490}, 'val': {'null_targets': 0, 'zero_demand': 271111, 'positive_demand': 14465, 'min_demand': 0, 'mean_demand': 1.9716257668711656, 'max_demand': 434}, 'test': {'null_targets': 0, 'zero_demand': 271481, 'positive_demand': 14095, 'min_demand': 0, 'mean_demand': 1.9364862593495251, 'max_demand': 431}}
Written: {'train': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_TRAIN.parquet'), 'val': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/data/full/train_test_data/GOLD_1H_DEMAND_HEXAGON_7_VAL.parquet'), 'test': PosixPath('/Users/lennartjekel/dev/git/Group-3-AAA/da

In [4]:
df_split.head(10).collect()

datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,date,is_holiday,community_area,weather_station_distance_km,food_drink,landmark,shop,train_station,tmpc,relh,sknt,p01m,vsby,wind_dir_sin,wind_dir_cos,station_observed,weather_imputed,precipitation_missing,weather_rain,weather_snow,weather_fog_mist,weather_thunder,weather_freezing,precipitation_trace,weather_qc_corrected,skyc1_CLR,skyc1_FEW,skyc1_SCT,skyc1_BKN,skyc1_OVC,skyc1_VV,weather_station_MDW,weather_station_ORD,weather_station_IGQ,trip_count,trip_seconds_sum,trip_seconds_mean,trip_seconds_min,trip_seconds_max,trip_miles_sum,trip_miles_mean,trip_miles_min,trip_miles_max,fare_sum,fare_mean,fare_min,fare_max,tips_sum,tips_mean,tips_min,tips_max,tolls_sum,tolls_mean,tolls_min,tolls_max,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type
datetime[μs],i8,i8,i8,f64,f64,f64,f64,f64,f64,date,i8,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,u32,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
2025-01-09 04:00:00,1,4,4,0.0,1.0,0.433884,-0.900969,0.866025,0.5,2025-01-09,0,72,10.128867,24.0,4.0,6.0,4.0,-9.027778,62.6275,4.0,0.0,10.0,-0.378917,-0.914138,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,1597,1597.0,1597,1597,9.74,9.74,9.74,9.74,33.62,33.62,33.62,33.62,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.62,33.62,33.62,33.62,"""Mobile"""
2025-01-11 08:00:00,1,6,8,0.0,1.0,-0.974928,-0.222521,0.866025,-0.5,2025-01-11,0,72,10.128867,24.0,4.0,6.0,4.0,-0.833333,71.9375,7.0,0.0,10.0,-0.977327,-0.085505,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,4,5101,1275.25,677,1870,19.8,4.95,1.27,9.03,68.75,17.1875,9.75,24.75,4.0,1.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,72.75,18.1875,9.75,24.75,"""Prcard"""
2025-02-15 08:00:00,2,6,8,0.5,0.866025,-0.974928,-0.222521,0.866025,-0.5,2025-02-15,0,72,10.128867,24.0,4.0,6.0,4.0,0.833333,72.1925,2.25,0.0,10.0,-0.163176,-0.059391,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,8,11809,1476.125,809,2306,66.45,8.30625,0.25,12.37,194.75,24.34375,6.0,33.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.125,0.0,1.0,195.75,24.46875,6.0,34.0,"""Prcard"""
2025-02-14 04:00:00,2,5,4,0.5,0.866025,-0.433884,-0.900969,0.866025,0.5,2025-02-14,0,72,10.128867,24.0,4.0,6.0,4.0,-13.888889,66.685,4.25,0.0,10.0,-0.548213,-0.786357,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-03-27 12:00:00,3,5,12,0.866025,0.5,-0.433884,-0.900969,1.2246e-16,-1.0,2026-03-27,0,72,10.128867,24.0,4.0,6.0,4.0,4.444444,48.8925,7.5,0.0,10.0,0.128917,0.394301,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,2,4462,2231.0,1818,2644,19.24,9.62,8.46,10.78,61.0,30.5,30.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,61.0,30.5,30.0,31.0,"""Prcard"""
2026-03-01 16:00:00,3,7,16,0.866025,0.5,-0.781831,0.62349,-0.866025,-0.5,2026-03-01,0,72,10.128867,24.0,4.0,6.0,4.0,-1.527778,48.3575,11.5,0.0,10.0,0.859447,0.496202,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,3,8717,2905.666667,1975,4200,46.25,15.416667,10.68,23.83,130.62,43.54,30.0,65.87,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,130.62,43.54,30.0,65.87,"""Prcard"""
2026-04-21 00:00:00,4,2,0,1.0,6.1232e-17,0.781831,0.62349,0.0,1.0,2026-04-21,0,72,10.128867,24.0,4.0,6.0,4.0,10.277778,43.29,8.75,0.0,10.0,-0.086824,-0.992404,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""No trips"""
2026-01-28 12:00:00,1,3,12,0.0,1.0,0.974928,-0.222521,1.2246e-16,-1.0,2026-01-28,0,72,10.128867,24.0,4.0,6.0,4.0,-8.75,43.7275,13.5,0.0,10.0,-0.973529,0.215741,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,4,8518,2129.5,1574,2640,51.82,12.955,8.6,16.23,146.5,36.625,26.0,45.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.25,0.0,1.0,147.5,36.875,26.0,46.5,"""Prcard"